In [0]:
USE databricks_ws.silver;

In [0]:
SELECT * FROM databricks_ws.silver.movie_casts;

movie_id,person_id,character_name,file_date,ingestion_date,environment
285,85,Captain Jack Sparrow,2024-12-16,2026-11-12T00:22:53.114651Z,Production
285,114,Will Turner,2024-12-16,2026-11-12T00:22:53.114651Z,Production
285,116,Elizabeth Swann,2024-12-16,2026-11-12T00:22:53.114651Z,Production
285,1640,William Bootstrap Bill Turner,2024-12-16,2026-11-12T00:22:53.114651Z,Production
285,1619,Captain Sao Feng,2024-12-16,2026-11-12T00:22:53.114651Z,Production
285,2440,Captain Davy Jones,2024-12-16,2026-11-12T00:22:53.114651Z,Production
285,118,Captain Hector Barbossa,2024-12-16,2026-11-12T00:22:53.114651Z,Production
285,1709,Admiral James Norrington,2024-12-16,2026-11-12T00:22:53.114651Z,Production
285,2449,Joshamee Gibbs,2024-12-16,2026-11-12T00:22:53.114651Z,Production
285,2441,Lord Cutler Beckett,2024-12-16,2026-11-12T00:22:53.114651Z,Production


In [0]:
CREATE TABLE IF NOT EXISTS databricks_ws.gold.bridge_movie_casts
USING DELTA
AS
SELECT
    movie_id,
    person_id,
    character_name
FROM databricks_ws.silver.movie_casts
WHERE 1 = 0;

num_affected_rows,num_inserted_rows


In [0]:
MERGE INTO databricks_ws.gold.bridge_movie_casts AS tgt
USING (
    SELECT
        c.movie_id,
        c.person_id,
        c.character_name
    FROM databricks_ws.silver.movie_casts c
    INNER JOIN databricks_ws.gold.dim_movies m
        ON c.movie_id = m.movie_id
    WHERE c.file_date = :file_date
       OR EXISTS (
           SELECT 1
           FROM databricks_ws.silver.movies sm
           WHERE sm.movie_id = c.movie_id
             AND sm.file_date = :file_date
       )
) AS src
ON tgt.movie_id = src.movie_id AND tgt.person_id = src.person_id

WHEN MATCHED THEN
    UPDATE SET *
WHEN NOT MATCHED THEN
    INSERT *;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
4933,4933,0,0


In [0]:
SELECT * FROM databricks_ws.gold.bridge_movie_casts;

movie_id,person_id,character_name
285,85,Captain Jack Sparrow
285,114,Will Turner
285,116,Elizabeth Swann
285,1640,William Bootstrap Bill Turner
285,1619,Captain Sao Feng
285,2440,Captain Davy Jones
285,118,Captain Hector Barbossa
285,1709,Admiral James Norrington
285,2449,Joshamee Gibbs
285,2441,Lord Cutler Beckett
